# Lending Club EDA — Data Understanding & Structural Profiling

**Purpose**
- Confirm table row counts at every pipeline stage
- Split every column in `windowed` into numeric vs. categorical (by cast-success rate)
- Profile null% and cardinality for every column

**Column count convention (used throughout)**
- `raw_mat` carries the **151** raw Lending Club columns
- `matured` and `windowed` add one column — `is_bad`, the modeling target built at ingestion — for **152**
- Cells 2, 3 and 6 profile `windowed`, so they report **152**; that is the 151 raw columns plus `is_bad`, not a discrepancy

**Where this fits**

| | |
|---|---|
| Position | 1 of 14 EDA notebooks under `notebooks/02_eda/` |
| Data access | Read-only against `data/02_interim/lendingclub.duckdb` |
| Cleaning | None here — happens later, in `notebooks/03_data_cleaning/` |
| Convention | Every code cell has markdown before it (what/why/how) and after it (what the output means, what's next) |

## Cell map

| # | Step |
|---|---|
| 1 | Connect, confirm row counts (raw / matured / windowed) |
| 2 | Numeric vs. categorical split, all 152 `windowed` columns (151 raw + `is_bad`) |
| 3 | Eyeball full column lists — sample values, cardinality |
| 4 | Tag date and time-period columns |
| 5 | Flag ambiguous columns for the cleaning stage, by row number |
| 6 | Full null% / cardinality profile (`SUMMARIZE`) |
| 7 | Check whether high-missingness columns carry signal |
| 8 | Append this notebook's findings to the field-treatment ledger |

**Scope**
- Profiles all 152 columns of `windowed` — the 151 raw columns plus the `is_bad` target added at ingestion — none picked or dropped yet
- Cell 8 starts a cumulative field-treatment ledger (`field_treatment_ledger.csv`) that notebooks 01–14 build up and `03_data_cleaning` executes — its rows are proposals, not enacted drops
- Starting with `02_eda/02_data_quality_integrity.ipynb`, every later notebook narrows to a fixed, named shortlist (see that notebook's cell 1) — that shortlist, not this notebook, drives the rest of the EDA suite

## Cell 1 — Connect & confirm row counts

- Open the interim DuckDB file read-only
- Confirm `raw_mat`, `matured`, `windowed` row counts match what the ingestion notebook produced — a sanity check before profiling anything

**Answers:** do the three staged tables have the row counts the ingestion notebook produced?

In [9]:
# Cell 1 -- open the database and check the three staged tables are the size we expect.

import os
import sys

import pandas as pd

# Make the shared helper in notebooks/_shared importable, then open the DB read-only.
sys.path.insert(0, os.path.abspath("../_shared"))
from nb_setup import connect

con, ASSETS_TABLES, ASSETS_PLOTS = connect()   # read-only connection; also creates the asset folders

# The pipeline builds three tables, in order: raw_mat -> matured -> windowed.
# Print each table's row count and column count so we can compare them to the ingestion notebook.
staged_tables = ["raw_mat", "matured", "windowed"]

for table_name in staged_tables:
    row_count = con.sql(f"SELECT count(*) FROM {table_name}").fetchone()[0]
    column_count = len(con.sql(f"DESCRIBE {table_name}").fetchall())
    print(f"{table_name:9s} -> {row_count:>10,} rows | {column_count} columns")

# `matured` and `windowed` should be the raw columns plus one extra: the modeling target.
# Compare the column name sets to see exactly what was added.
raw_column_names = set(con.sql("DESCRIBE raw_mat").df()["column_name"])
windowed_column_names = set(con.sql("DESCRIBE windowed").df()["column_name"])
added_column_names = sorted(windowed_column_names - raw_column_names)

print(f"\ncolumns added after raw_mat: {added_column_names}")

raw_mat   ->  2,260,701 rows | 151 columns
matured   ->  1,348,099 rows | 152 columns
windowed  ->  1,195,879 rows | 152 columns

columns added after raw_mat: ['is_bad']


**Result**

| Table | Rows | Columns |
|---|---|---|
| `raw_mat` | 2,260,701 | 151 |
| `matured` | 1,348,099 | 152 |
| `windowed` | 1,195,879 | 152 |

- 151 raw columns confirmed on `raw_mat` — matches the ingestion notebook's output
- `matured` and `windowed` carry those 151 plus `is_bad` (the modeling target defined at ingestion) = 152 — this is why the profiling cells below report 152, not 151
- Every downstream notebook works from a subset of these

**Next:** split columns into numeric vs. categorical — nothing arrived typed (everything loaded as `VARCHAR` on purpose, so nothing gets silently mis-cast).

## Cell 2 — Numeric vs. categorical, by cast-success rate

- For every column in `windowed`, check what fraction of non-null values `TRY_CAST`s to `DOUBLE`
- More reliable than trusting column names; flags genuinely ambiguous columns

**Answers:** how many columns are numeric, how many categorical, and are any ambiguous?

In [10]:
# Cell 2 -- work out whether each column is really a number or really text.
#
# Every column was loaded as text (VARCHAR) on purpose. To find its true type we try to
# convert each value to a number and measure how often that succeeds ("cast rate").

# 1. Get the name of every column in the `windowed` table.
column_names = con.sql("DESCRIBE windowed").df()["column_name"].tolist()

# 2. For each column, ask DuckDB four counts in a single query.
profile_records = []

for column in column_names:
    counts_query = f'''
        SELECT
            count(*)                               AS n_rows,       -- rows in the table
            count("{column}")                      AS n_filled,     -- rows where this column is not NULL
            count(DISTINCT "{column}")             AS n_distinct,   -- how many different values it has
            count(TRY_CAST("{column}" AS DOUBLE))  AS n_numeric     -- how many values convert to a number
        FROM windowed
    '''
    n_rows, n_filled, n_distinct, n_numeric = con.sql(counts_query).fetchone()

    # Share of filled values that look numeric. Guard against divide-by-zero for all-NULL columns.
    cast_rate = n_numeric / n_filled if n_filled else 0.0
    pct_notnull = n_filled / n_rows

    profile_records.append((column, pct_notnull, n_distinct, cast_rate))

# 3. Put the per-column numbers into a DataFrame.
type_df = pd.DataFrame(
    profile_records,
    columns=["column", "pct_notnull", "n_distinct", "cast_rate"],
)


# 4. Turn each cast_rate into a plain-language label.
def label_from_cast_rate(cast_rate):
    if cast_rate > 0.95:            # nearly everything converts -> treat as numeric
        return "numeric"
    if cast_rate < 0.05:            # nearly nothing converts -> treat as text / category
        return "categorical/text"
    return "mixed"                  # somewhere in between -> a human should look


type_df["inferred_type"] = type_df["cast_rate"].apply(label_from_cast_rate)

# 5. Report the counts, and list any "mixed" columns.
print(type_df["inferred_type"].value_counts())
print()
print("mixed columns (worth a manual look):")
print(type_df[type_df["inferred_type"] == "mixed"].to_string(index=False))

inferred_type
numeric             114
categorical/text     38
Name: count, dtype: int64

mixed columns (worth a manual look):
Empty DataFrame
Columns: [column, pct_notnull, n_distinct, cast_rate, inferred_type]
Index: []


**Result**

| Type | Count |
|---|---|
| Numeric | 114 |
| Categorical/text | 38 |
| Mixed (ambiguous) | 0 |

- 114 + 38 = 152 — every column in `windowed` (the 151 raw columns + `is_bad`); `is_bad` casts cleanly as 0/1 and lands in the numeric bucket, but it's the target, not a feature (see cell 3)
- No columns landed in the "mixed" bucket — confirms the type assumptions the cleaning pipeline and every other EDA notebook rely on

**Next:** the cast-rate threshold is mechanical — it can still misclassify a column a human would recognize immediately (e.g. a numeric-looking ID or code). Eyeballing the actual lists, then flagging anything that looks wrong, catches what the automated split can't.

## Cell 3 — Eyeball the full column lists

- The 95%/5% cast-rate threshold is mechanical — it can't tell a genuine numeric measure from a numeric-looking identifier or code, and cardinality alone doesn't either (a rare-event count can have as few distinct values as a real code)
- Printing every column with a few actual sample values and its distinct-value count lets a human check both at once, before moving on

**Answers:** looking at real sample values and cardinality, does anything in the numeric or categorical bucket look miscategorized?

In [11]:
# Cell 3 -- print every column with a few real example values, so a person can spot
# anything the automatic test in Cell 2 got wrong (for example an ID made of digits).

# Split the column names into the two buckets from Cell 2, sorted alphabetically.
numeric_cols = sorted(type_df.loc[type_df["inferred_type"] == "numeric", "column"])
categorical_cols = sorted(type_df.loc[type_df["inferred_type"] == "categorical/text", "column"])

# Longest example value to show in the table; longer values are trimmed with a trailing "...".
MAX_SAMPLE_WIDTH = 35


def shorten(value):
    """Return `value` as a string, trimmed to MAX_SAMPLE_WIDTH characters."""
    text = str(value)
    if len(text) <= MAX_SAMPLE_WIDTH:
        return text
    return text[:MAX_SAMPLE_WIDTH - 3] + "..."


def pick_example_values(column, how_many=4):
    """Return `how_many` distinct, non-NULL example values for `column`.

    - Sampled from a DISTINCT sub-query, so the examples differ from each other.
    - No random seed, so each run shows different examples.
    - If the column has fewer than `how_many` distinct values, the list is padded
      with "" (empty string) -- itself a useful hint that the column is low-variety.
    """
    sample_query = f'''
        SELECT value
        FROM (SELECT DISTINCT "{column}" AS value FROM windowed WHERE "{column}" IS NOT NULL)
        USING SAMPLE {how_many} ROWS
    '''
    values = con.sql(sample_query).df()["value"].tolist()

    # Pad short lists to a fixed length so every preview row has the same shape.
    values = values + [""] * (how_many - len(values))

    return [shorten(value) for value in values]


def build_preview_table(column_list):
    """Build one row per column: name, four example values, distinct count, is_ambiguous flag."""
    preview_records = []

    for column in column_list:
        n_distinct = con.sql(f'SELECT count(DISTINCT "{column}") FROM windowed').fetchone()[0]
        example_1, example_2, example_3, example_4 = pick_example_values(column)
        preview_records.append(
            (column, example_1, example_2, example_3, example_4, n_distinct, "No")
        )

    return pd.DataFrame(
        preview_records,
        columns=["column", "sample_1", "sample_2", "sample_3", "sample_4",
                 "n_distinct", "is_ambiguous"],
    )


numeric_preview_df = build_preview_table(numeric_cols)
categorical_preview_df = build_preview_table(categorical_cols)

print(f"numeric columns ({len(numeric_preview_df)}):")
print(numeric_preview_df.to_string())
print()
print(f"categorical columns ({len(categorical_preview_df)}):")
print(categorical_preview_df.to_string())

numeric columns (114):
                                         column            sample_1          sample_2        sample_3          sample_4  n_distinct is_ambiguous
0                                acc_now_delinq                 0.0               6.0             3.0               1.0           8           No
1                          acc_open_past_24mths                31.0              11.0            37.0              17.0          55           No
2                                      all_util                79.0               6.0            61.0             168.0         169           No
3                                    annual_inc             39388.0          100076.0         48630.0           75587.0       59182           No
4                              annual_inc_joint            189500.0           38072.0         69600.0           92110.0        4052           No
5                                   avg_cur_bal             25017.0           13713.0         13325.0      

**Result**

Two columns in the numeric bucket look miscategorized once actual values and cardinality are visible:

| Column | n_distinct | Why it's not really numeric |
|---|---|---|
| `id` | ~1.2M (nearly all unique) | A loan identifier — casts to `DOUBLE` because it's stored as digits, but it's a label, not a magnitude |
| `policy_code` | 1 | A single-valued category code in this population, not a measured quantity |

One more worth naming explicitly: `is_bad` also sits in the numeric bucket (it casts cleanly, 0/1) — but it's the modeling target defined in the ingestion notebook, not a candidate feature. It's excluded from feature consideration by definition, not because anything about it needs flagging.

Everything else, including the other low-`n_distinct` numeric columns (e.g. `num_tl_30dpd`, `inq_last_6mths`, `acc_now_delinq`), reads as expected: genuine counts of a rare event, not category codes — small range because the event is rare, not because the field is secretly categorical.

**Next:** a few categorical columns are also worth a closer look — some are dates or durations in disguise.

## Cell 4 — Tag date and time-period columns

- Some categorical columns aren't general text — they're dates or durations, just not parsed as such (parsing/casting is a cleaning-stage decision, not an EDA one)
- Printing actual raw values before tagging, so the tag is evidenced, not assumed
- This is additive: `inferred_type` (numeric/categorical, from the cast-rate test) stays exactly as computed — `domain_type` is a separate note layered on top

**Answers:** which categorical columns are actually dates, which are durations, and what does their raw format actually look like?

In [12]:
# Cell 4 -- a few "categorical" columns are really dates or time spans stored as text.
# Show raw examples first (evidence), then add a separate, additive `domain_type` label.
# The Cell 2 `inferred_type` stays as the mechanical source of truth.

# Columns whose values look like "May-2016" (a month and a year).
DATE_LIKE = [
    "earliest_cr_line",
    "issue_d",
    "last_credit_pull_d",
    "last_pymnt_d",
    "sec_app_earliest_cr_line",
]

# Columns whose values are a length of time written as text, e.g. "36 months", "10+ years".
TIME_PERIOD_LIKE = ["emp_length", "term"]

# Print the first few real values of each, so the labels below are backed by evidence.
for column in DATE_LIKE + TIME_PERIOD_LIKE:
    example_values = con.sql(
        f'SELECT "{column}" FROM windowed WHERE "{column}" IS NOT NULL LIMIT 6'
    ).df()[column].tolist()
    print(f"{column}: {example_values}")

# Add the `domain_type` column: default everything to "categorical", then relabel the two groups.
categorical_preview_df["domain_type"] = "categorical"
categorical_preview_df.loc[categorical_preview_df["column"].isin(DATE_LIKE), "domain_type"] = "date"
categorical_preview_df.loc[categorical_preview_df["column"].isin(TIME_PERIOD_LIKE), "domain_type"] = "time_period"

print()
print(categorical_preview_df.to_string())

earliest_cr_line: ['May-2002', 'Dec-2010', 'Jan-2006', 'Jul-1994', 'Aug-1997', 'Jun-1989']
issue_d: ['May-2016', 'May-2016', 'May-2016', 'May-2016', 'May-2016', 'May-2016']
last_credit_pull_d: ['Mar-2019', 'Feb-2017', 'Jan-2019', 'Nov-2016', 'Mar-2019', 'Feb-2019']
last_pymnt_d: ['Feb-2019', 'Feb-2017', 'Jan-2019', 'Oct-2016', 'Jan-2017', 'Sep-2018']
sec_app_earliest_cr_line: ['Aug-1990', 'Jul-2008', 'Jul-2007', 'Apr-2014', 'Apr-2004', 'Mar-2004']
emp_length: ['4 years', '3 years', '9 years', '10+ years', '2 years', '10+ years']
term: [' 36 months', ' 36 months', ' 36 months', ' 36 months', ' 36 months', ' 36 months']

                       column                             sample_1                             sample_2                             sample_3                             sample_4  n_distinct is_ambiguous  domain_type
0                  addr_state                                   LA                                   AZ                                   NJ                 

**Result**

| Group | Columns | Raw format |
|---|---|---|
| `date` | `earliest_cr_line`, `issue_d`, `last_credit_pull_d`, `last_pymnt_d`, `sec_app_earliest_cr_line` | `Mon-YYYY` (month abbreviation + year, no day) |
| `time_period` | `emp_length`, `term` | Free-text duration (`"10+ years"`, `"36 months"`) |

All five date-like columns share the same raw text format — consistent with being genuine dates that just haven't been parsed yet. None of this changes `inferred_type`; both groups still correctly read as `categorical/text` by the cast-rate test, since none of these strings cast to `DOUBLE`.

**Next:** record `id` and `policy_code` — the two genuinely miscategorized columns — somewhere the cleaning notebook can actually check.

## Cell 5 — Flag ambiguous columns for the cleaning stage, by row number

- A markdown note here is easy to miss; a file the cleaning notebook actually reads is not
- Referencing rows from the tables printed in cell 3 (by table + row number) is less error-prone than retyping column names by hand
- Writes the flagged column + reason to `eda01_ambiguous_overrides.csv`, so `03_data_cleaning/01_cleaning_and_feature_prep.ipynb` can act on it instead of relying on the cast-rate split alone

**Answers:** which columns need to be handled by note, not by cast-rate, and why?

In [13]:
# Cell 5 -- record the columns that passed the numeric test in Cell 2 but are NOT real
# numeric features. Point at them by their row number in the Cell 3 preview tables.

# Each entry: (which Cell 3 table, row number in that table, reason for flagging).
FLAGGED_ROWS = [
    ("numeric", 25, "loan identifier, not a numeric feature -- exclude from modeling despite passing the cast-rate test"),
    ("numeric", 77, "single-valued category code in this population, not a measured quantity -- treat as categorical"),
]

# Map the short table name used above to the actual preview DataFrame.
preview_by_name = {
    "numeric": numeric_preview_df,
    "categorical": categorical_preview_df,
}

flagged_overrides = []

for table_name, row_number, reason in FLAGGED_ROWS:
    preview_df = preview_by_name[table_name]

    # Read the column name from that row, and mark the row as ambiguous in the preview table.
    column_name = preview_df.loc[row_number, "column"]
    preview_df.loc[row_number, "is_ambiguous"] = "Yes"

    flagged_overrides.append((column_name, reason))

# Save the flags so the cleaning notebook can act on them.
overrides_df = pd.DataFrame(flagged_overrides, columns=["column", "note"])
overrides_df.to_csv(os.path.join(ASSETS_TABLES, "eda01_ambiguous_overrides.csv"), index=False)

print(f"ambiguous-column overrides recorded: {len(overrides_df)}")
print(overrides_df.to_string(index=False))

ambiguous-column overrides recorded: 2
     column                                                                                               note
         id loan identifier, not a numeric feature -- exclude from modeling despite passing the cast-rate test
policy_code    single-valued category code in this population, not a measured quantity -- treat as categorical


**Result**

| column | note |
|---|---|
| `id` | loan identifier, not a numeric feature -- exclude from modeling despite passing the cast-rate test |
| `policy_code` | single-valued category code in this population, not a measured quantity -- treat as categorical |

Saved to `eda01_ambiguous_overrides.csv` — the cleaning notebook should check this file and treat these two by their note, not by cast-rate alone. This list is meant to grow: any later notebook that spots another miscategorized column should add it here rather than handling it locally.

**Next:** full null%/cardinality profile across every column.

## Cell 6 — Full null% / cardinality profile, every column

- DuckDB's `SUMMARIZE` gives a one-pass null%/min/max/approx-distinct profile per column, in a single query
- Run against `windowed`, covering all 152 columns (the 151-column raw schema + the `is_bad` target)

**Answers:** which columns are the most/least populated, and are any fully null?

In [14]:
# Cell 6 -- one built-in DuckDB command, SUMMARIZE, profiles every column at once:
# its storage type, how much is missing, and roughly how many distinct values it holds.

summary_df = con.sql("SUMMARIZE windowed").df()

# Keep the columns we care about; convert the missing-percentage text to a real number.
summary_df["null_pct"] = summary_df["null_percentage"].astype(float)

summ_sorted = (
    summary_df[["column_name", "column_type", "null_pct", "approx_unique"]]
    .sort_values("null_pct", ascending=False)
    .reset_index(drop=True)
)

# Print the full profile (every column, most-missing first) and save it for later notebooks.
print(summ_sorted.to_string(index=False))
summ_sorted.to_csv(os.path.join(ASSETS_TABLES, "eda01_summ_sorted.csv"), index=False)

# Two headline counts.
n_fully_populated = (summ_sorted["null_pct"] == 0).sum()
n_fully_empty = (summ_sorted["null_pct"] == 100).sum()

print()
print(f"columns with 0% nulls: {n_fully_populated} of {len(summ_sorted)}")
print(f"columns 100% null: {n_fully_empty}")

                               column_name column_type  null_pct  approx_unique
                                 member_id     VARCHAR    100.00              0
                              next_pymnt_d     VARCHAR    100.00              1
orig_projected_additional_accrued_interest     VARCHAR     99.69           3433
       sec_app_mths_since_last_major_derog     VARCHAR     99.65             93
                           hardship_length     VARCHAR     99.52              1
                           hardship_amount     VARCHAR     99.52           4136
                           hardship_status     VARCHAR     99.52              1
                             deferral_term     VARCHAR     99.52              1
                         hardship_end_date     VARCHAR     99.52             27
                           hardship_reason     VARCHAR     99.52              9
                       hardship_start_date     VARCHAR     99.52             27
                             hardship_ty

**Result**

| Metric | Value |
|---|---|
| Columns profiled | 152 (151 raw + `is_bad`) |
| Fully populated columns | 80 of 152 |
| Fully null columns | 2 (`member_id`, `next_pymnt_d`) |

- `member_id` is scrubbed by Lending Club before publication — dead weight, not signal
- The highest-missingness band (~99.5%+) is entirely hardship/settlement fields — worth understanding on their own terms, not dismissing as "mostly empty"

**Next:** whether that missingness is itself informative, or just structural noise.

## Cell 7 — Does high missingness carry signal?

- A field that's 99.5% null isn't automatically useless — if *whether* it's populated correlates with the outcome, the missingness itself is a feature
- Checking this directly rather than assuming it either way

**Answers:** do loans with a populated hardship/settlement record have a different bad rate than loans without one?

In [15]:
# Cell 7 -- a column that is almost always empty can still be useful IF being filled in
# lines up with loans going bad. Check that for the hardship and settlement fields.

# 1. Bad rate for loans that HAVE a hardship record vs. loans that do not.
#    `hardship_type IS NULL` is TRUE when there is no hardship record.
hardship_signal = con.sql("""
    SELECT
        (hardship_type IS NULL)  AS hardship_type_is_null,
        count(*)                 AS n,
        round(avg(is_bad), 3)    AS bad_rate
    FROM windowed
    GROUP BY 1
    ORDER BY 1
""").df()

print("bad rate by whether hardship_type is populated:")
print(hardship_signal.to_string(index=False))
print()

# 2. Same idea for the debt-settlement flag (its values are 'Y' / 'N').
settlement_signal = con.sql("""
    SELECT
        debt_settlement_flag,
        count(*)               AS n,
        round(avg(is_bad), 3)  AS bad_rate
    FROM windowed
    GROUP BY 1
    ORDER BY 1
""").df()

print("bad rate by debt_settlement_flag:")
print(settlement_signal.to_string(index=False))
print()

# 3. Does `hardship_flag` actually vary here, or is it always the same value?
hardship_flag_values = con.sql("SELECT DISTINCT hardship_flag FROM windowed").df()
print(f"distinct hardship_flag values in windowed: {hardship_flag_values['hardship_flag'].tolist()}")

# Save both signal tables for later notebooks.
hardship_signal.to_csv(os.path.join(ASSETS_TABLES, "eda01_hardship_signal.csv"), index=False)
settlement_signal.to_csv(os.path.join(ASSETS_TABLES, "eda01_settlement_signal.csv"), index=False)

bad rate by whether hardship_type is populated:
 hardship_type_is_null       n  bad_rate
                 False    5726     0.705
                  True 1190153     0.203

bad rate by debt_settlement_flag:
debt_settlement_flag       n  bad_rate
                   N 1163542     0.183
                   Y   32337     1.000

distinct hardship_flag values in windowed: ['N']


**Result**

| Group | Populated | n | Bad rate |
|---|---|---|---|
| Hardship record exists (`hardship_type` not null) | Yes | 5,726 | 70.5% |
| No hardship record | No | 1,190,153 | 20.3% |
| Debt settlement flag = Y | Yes | 32,337 | 100.0% |
| Debt settlement flag = N | No | 1,163,542 | 18.3% |

- `hardship_flag` is a dead end for this population — every row in `windowed` shows `N` (it marks a loan *currently* on an active hardship plan; a matured/finished loan is never "currently" anything)
- `hardship_type IS NULL` is the field that actually carries the history

**What these fields are**

| Field group | What it represents | Populated when |
|---|---|---|
| Hardship (`hardship_type`, `hardship_length`, `hardship_amount`, `hardship_start_date`/`hardship_end_date`, ...) | Temporary payment-relief program — reduced or paused payments for a borrower in financial distress | Loan entered a hardship plan |
| Settlement (`settlement_status`, `settlement_amount`, `settlement_percentage`, `settlement_term`, ...) | Negotiated payoff for less than the full balance, via a third-party settlement company | Loan is already severely delinquent |

Both are event-triggered — populated only for the subset of loans that entered that specific process, not the general population.

**Is the missingness itself a signal?**
- Yes, and a strong one
- Hardship record present → 3.5x the base bad rate (70.5% vs. 20.3%)
- Debt settlement flag = Y → 100.0% bad, by construction — settlement is a workout for loans already failing, not a trait observed before the outcome

**Should they be used in modeling?**
- Not as raw predictors — this is a leakage question, not a missingness question
- Both fields only exist because a loan started going bad; they document the outcome unfolding, not a borrower characteristic knowable at origination
- A model trained on `debt_settlement_flag` would just relearn "settled loans are bad" (already true by definition) — and a brand-new loan has no settlement history yet at scoring time anyway
- Same logic applies to `hardship_type` and its associated fields

**What they're good for instead, and when**

| Use | Stage | Notes |
|---|---|---|
| Binary "ever had a hardship/settlement event" indicator (from the null pattern) | Post-hoc, not origination-time | Collections prioritization, loss-given-default modeling on already-troubled loans |
| Exclude from PD feature set, with leakage reasoning documented | `03_data_cleaning/01_cleaning_and_feature_prep.ipynb` | Not just dropped for "too many nulls" |
| Revisit if target needs more granularity than binary `is_bad` | `02_eda/07_target_outcome_objective.ipynb` | Only if that need arises |

**How they'd be treated if ever used**
- The null pattern itself is the feature (`hardship_type IS NULL` as a 0/1 flag) — not the raw amount/date fields
- Those raw fields are only meaningful conditional on the event having happened; imputing them for the 99.5% of loans where nothing happened would fabricate values with no real meaning

## Cell 8 — Append findings to the field-treatment ledger

- One cumulative file — `data/04_assets/tables/field_treatment_ledger.csv`, one row per field
- Every EDA notebook (01–14) upserts the rows its own analysis touched; after notebook 14 the file is a complete per-field treatment plan
- `03_data_cleaning/01_cleaning_and_feature_prep.ipynb` reads that final file and executes it — the ledger is the contract between EDA and cleaning
- Self-contained read-modify-write, keyed on `column`: re-running any notebook is idempotent
- Every row is `status = "proposed"` and cites the cell it came from — cleaning is where a row becomes `confirmed` and is enacted; **nothing is dropped in this notebook**

**Schema**

| Field | Meaning |
|---|---|
| `column` | field name |
| `inferred_type` | numeric / categorical/text — mechanical, from cell 2 |
| `domain_type` | numeric / categorical / date / time_period — from cell 4 |
| `action` | `keep` · `clean` (dtype cast) · `transform` (parse/reshape) · `drop` · `exclude_from_pd_features` (retain in data, not a PD predictor) · `review` (a later notebook decides) |
| `treatment_detail` | concrete operation for the cleaning notebook |
| `basis` | which analysis produced the row |
| `rationale` | why |
| `status` | `proposed` (EDA) → `confirmed` (cleaning) |
| `last_updated_by` | notebook id of the most recent writer |

**Answers:** what does eda01's analysis imply for each field, and where is that written down for the cleaning stage?

In [16]:
# Cell 8 -- write this notebook's conclusions into a shared "field treatment ledger".
#
# The ledger is ONE csv with ONE row per column. Every EDA notebook (01-14) updates the
# rows it has something to say about. The cleaning notebook (03_data_cleaning) then reads
# the finished ledger and does exactly what it says.
#
# Safe to re-run: rows are matched by column name and replaced, never appended twice.

# --- settings ---------------------------------------------------------------
LEDGER_PATH = os.path.join(ASSETS_TABLES, "field_treatment_ledger.csv")
NOTEBOOK_ID = "eda01"

LEDGER_COLUMNS = [
    "column",            # the field name
    "inferred_type",     # numeric / categorical/text                    (from Cell 2)
    "domain_type",       # numeric / categorical / date / time_period    (from Cell 4)
    "action",            # what cleaning should do -- see the choices below
    "treatment_detail",  # the concrete step to take
    "basis",             # which analysis this row came from
    "rationale",         # why
    "status",            # "proposed" here; the cleaning notebook sets "confirmed"
    "last_updated_by",   # id of the notebook that last wrote this row
]

# Allowed values for "action":
#   keep                     -> use the column as-is
#   clean                    -> fix the data type only (text -> number)
#   transform                -> reshape the value (parse a date; "36 months" -> 36)
#   drop                     -> remove the column entirely
#   exclude_from_pd_features -> keep the column, but never use it as a PD model input
#   review                   -> a later notebook must decide


# --- step 1: one baseline row for every column we profiled ----------------
# `domain_type` was only worked out for categorical columns (Cell 4); look it up per column.
domain_type_by_column = dict(
    zip(categorical_preview_df["column"], categorical_preview_df["domain_type"])
)

ledger_rows = {}   # column name -> dict of ledger fields

for _, profile in type_df.iterrows():
    column = profile["column"]
    is_numeric = profile["inferred_type"] == "numeric"

    ledger_rows[column] = {
        "column": column,
        "inferred_type": profile["inferred_type"],
        "domain_type": domain_type_by_column.get(column, "numeric" if is_numeric else "categorical"),
        "action": "clean" if is_numeric else "keep",
        "treatment_detail": "cast text -> DOUBLE" if is_numeric else "keep as text / category",
        "basis": "eda01 Cell 2 (numeric cast-rate test)",
        "rationale": "loaded as text on purpose; the cast-rate test decides the real type",
        "status": "proposed",
        "last_updated_by": NOTEBOOK_ID,
    }


# --- step 2: specific overrides, each backed by a printed result above ----
# Every override lists only the fields it changes; the rest stay as the baseline row.
override_specs = []

# Cell 4: five columns are dates written as "May-2016".
for column in DATE_LIKE:
    override_specs.append({
        "column": column,
        "domain_type": "date",
        "action": "transform",
        "treatment_detail": "parse 'Mon-YYYY' text -> DATE (format '%b-%Y'); no day part",
        "basis": "eda01 Cell 4 (looked at raw values)",
        "rationale": "a real date, just stored as month-year text",
    })

# Cell 4: two columns are time spans written as text.
override_specs.append({
    "column": "term",
    "domain_type": "time_period",
    "action": "transform",
    "treatment_detail": "trim spaces, take the number of months (36 or 60), cast INT",
    "basis": "eda01 Cell 4",
    "rationale": "duration stored as text like ' 36 months'",
})
override_specs.append({
    "column": "emp_length",
    "domain_type": "time_period",
    "action": "transform",
    "treatment_detail": "map '< 1 year' -> 0, '10+ years' -> 10, 'N years' -> N; cast INT; decide missing handling in 03_data_cleaning",
    "basis": "eda01 Cell 4",
    "rationale": "duration text with open-ended buckets at both ends",
})

# Cell 5: columns that pass the numeric test but are not real numbers.
override_specs.append({
    "column": "id",
    "action": "keep",
    "treatment_detail": "keep as the row identifier; never use as a model feature",
    "basis": "eda01 Cell 5 (eda01_ambiguous_overrides.csv)",
    "rationale": "a loan ID -- looks numeric but it is only a label",
})
override_specs.append({
    "column": "policy_code",
    "action": "review",
    "treatment_detail": "treat as a category; only 1 distinct value here, so it may add nothing",
    "basis": "eda01 Cell 5 + Cell 3",
    "rationale": "a code, not a measured quantity",
})

# Cells 6-7: columns that are entirely empty or never change.
override_specs.append({
    "column": "member_id",
    "action": "drop",
    "treatment_detail": "drop the column",
    "basis": "eda01 Cell 6 (SUMMARIZE null %)",
    "rationale": "100% empty -- removed by Lending Club before publishing",
})
override_specs.append({
    "column": "next_pymnt_d",
    "action": "drop",
    "treatment_detail": "drop the column",
    "basis": "eda01 Cell 6",
    "rationale": "100% empty in windowed -- finished loans have no next payment",
})
override_specs.append({
    "column": "hardship_flag",
    "action": "drop",
    "treatment_detail": "drop the column",
    "basis": "eda01 Cell 7",
    "rationale": "always 'N' here (never varies); marks a currently-active plan, which finished loans cannot have",
})

# The modeling target -- not a feature.
override_specs.append({
    "column": "is_bad",
    "action": "keep",
    "treatment_detail": "this is the target we predict, not an input",
    "basis": "ingestion notebook",
    "rationale": "defined as the 0/1 default target",
})

# Cell 7: hardship + settlement fields are only filled AFTER a loan is in trouble, so using
# them to predict default would be cheating (data leakage). Keep them, but not as PD inputs.
leakage_columns = [name for name in ledger_rows if name.startswith("hardship_")]
leakage_columns += [
    name for name in ledger_rows
    if name.startswith("settlement_") or name == "debt_settlement_flag"
]

for column in leakage_columns:
    override_specs.append({
        "column": column,
        "action": "exclude_from_pd_features",
        "treatment_detail": "keep the column; not a PD input; 'is it filled in?' can help LGD / collections work",
        "basis": "eda01 Cell 7 (bad rate by whether the field is filled)",
        "rationale": "only populated after a loan starts failing -- not known at application time",
    })


# --- step 3: apply each override on top of its baseline row --------------
for override in override_specs:
    column = override["column"]

    ledger_rows[column].update(override)          # change only the fields named in the override
    ledger_rows[column]["status"] = "proposed"
    ledger_rows[column]["last_updated_by"] = NOTEBOOK_ID

this_notebook_df = pd.DataFrame(list(ledger_rows.values()))[LEDGER_COLUMNS]


# --- step 4: merge into the shared ledger file and save ----------------
if os.path.exists(LEDGER_PATH):
    existing_df = pd.read_csv(LEDGER_PATH).astype(str)

    # Keep rows for columns this notebook did NOT touch, then add this notebook's rows.
    untouched_rows = existing_df[~existing_df["column"].isin(this_notebook_df["column"])]
    ledger_df = pd.concat([untouched_rows, this_notebook_df], ignore_index=True)
else:
    ledger_df = this_notebook_df

ledger_df = ledger_df.sort_values("column").reset_index(drop=True)
ledger_df.to_csv(LEDGER_PATH, index=False)

# --- report ----------------------------------------------------------
print(f"field_treatment_ledger.csv  ->  {len(ledger_df)} rows ({ledger_df['column'].nunique()} unique columns)")
print()
print("rows by action:")
print(ledger_df["action"].value_counts())
print()
print(ledger_df.to_string(index=False))

field_treatment_ledger.csv  ->  152 rows (152 unique columns)

rows by action:
action
clean                       103
keep                         21
exclude_from_pd_features     18
transform                     7
drop                          2
review                        1
Name: count, dtype: int64

                                    column    inferred_type domain_type                   action                                                                                              treatment_detail                                                  basis                                                                   rationale   status last_updated_by
                            acc_now_delinq          numeric     numeric                    clean                                                                                           cast text -> DOUBLE                  eda01 Cell 2 (numeric cast-rate test)         loaded as text on purpose; the cast-rate test decides the real

**Result**

- `field_treatment_ledger.csv` written — one row per profiled field, all `status = "proposed"`
- eda01's non-default entries:

| Group | Fields | action |
|---|---|---|
| Date parse (`Mon-YYYY` → DATE) | `earliest_cr_line`, `issue_d`, `last_credit_pull_d`, `last_pymnt_d`, `sec_app_earliest_cr_line` | `transform` |
| Duration parse (text → INT) | `term`, `emp_length` | `transform` |
| 100% null | `member_id`, `next_pymnt_d` | `drop` |
| Zero variance in `windowed` | `hardship_flag` | `drop` |
| Identifier / near-constant | `id`, `policy_code` | `keep` / `review` |
| Hardship + settlement (event-triggered leakage) | `hardship_*`, `settlement_*`, `debt_settlement_flag` | `exclude_from_pd_features` |
| `is_bad` | — | `keep` (target) |

- Every other column defaults to `clean` (numeric → cast to `DOUBLE`) or `keep` (text)

**Next:** `02_eda/02_data_quality_integrity.ipynb` — its findings upsert into this same file.